In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import re
import random
from pathlib import Path

import numpy as np
import pandas as pd

# -----------------------------
# CONFIG
# -----------------------------
DATA_ROOT = Path(r"/content/drive/MyDrive/ENT/Data")
PATIENT_XLSX = Path(r"/content/drive/MyDrive/ENT/Data/Patients_List_Updated_Final.xlsx")  # or wherever your xlsx is
OUT_DIR = DATA_ROOT / "_challenge2_artifacts"
OUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

RNG_SEED = 42
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.10, 0.20

# -----------------------------
# HELPERS
# -----------------------------
def infer_patient_id(path_str: str) -> str | None:
    """
    Tries to find PatientXXX in the path.
    Adjust regex if your naming differs.
    """
    m = re.search(r"(Patient\d{3})", path_str, flags=re.IGNORECASE)
    return m.group(1) if m else None

def infer_modality(path_str: str) -> str | None:
    """
    Best-effort modality inference from folder/file names.
    Adapt tokens to your dataset naming.
    """
    s = path_str.lower()
    # Common patterns; update if your naming uses different tokens
    if "nbi" in s or "narrow" in s:
        return "NBI"
    if "wle" in s or "white" in s or "wl" in s:
        return "WLE"
    return None

def stratified_patient_split(patients_df: pd.DataFrame, seed=42):
    """
    Patient-level stratified split on label_binary.
    """
    rng = random.Random(seed)
    benign = patients_df[patients_df["label_binary"] == 0]["Patient"].tolist()
    malignant = patients_df[patients_df["label_binary"] == 1]["Patient"].tolist()

    rng.shuffle(benign)
    rng.shuffle(malignant)

    def split_list(lst):
        n = len(lst)
        n_train = int(round(n * TRAIN_FRAC))
        n_val = int(round(n * VAL_FRAC))
        train = lst[:n_train]
        val = lst[n_train:n_train + n_val]
        test = lst[n_train + n_val:]
        return train, val, test

    b_tr, b_va, b_te = split_list(benign)
    m_tr, m_va, m_te = split_list(malignant)

    train = b_tr + m_tr
    val = b_va + m_va
    test = b_te + m_te

    rng.shuffle(train); rng.shuffle(val); rng.shuffle(test)
    return train, val, test

# -----------------------------
# 1) LOAD + CLEAN PATIENT TABLE
# -----------------------------
patients = pd.read_excel(PATIENT_XLSX)
patients = patients[patients["Patient"].notna()].copy()

# Create binary label
patients["label_binary"] = (patients["Type of Lesion"].str.lower() == "malignant").astype(int)

# Sanity checks
assert patients["Patient"].nunique() == len(patients), "Duplicate patient IDs in patient table."
print("Patients:", len(patients), "| Malignant:", patients["label_binary"].sum(), "| Benign:", (patients["label_binary"]==0).sum())

patients.to_csv(OUT_DIR / "patients_clean.csv", index=False)

# -----------------------------
# 2) SCAN IMAGES + BUILD MANIFEST
# -----------------------------
rows = []
for p in DATA_ROOT.rglob("*"):
    if p.is_file() and p.suffix.lower() in IMG_EXTS:
        pid = infer_patient_id(str(p))
        if pid is None:
            continue  # skip images we can't map to patient
        modality = infer_modality(str(p))
        rows.append({"filepath": str(p), "Patient": pid, "modality": modality})

manifest = pd.DataFrame(rows)
print("Images found (mappable):", len(manifest))
manifest.to_csv(OUT_DIR / "manifest_paths_only.csv", index=False)

# Join labels
manifest = manifest.merge(
    patients[["Patient", "label_binary", "Histopathology", "Leukoplakia", "Type of Lesion"]],
    on="Patient",
    how="inner"
)

# Remove any duplicates by path
manifest = manifest.drop_duplicates(subset=["filepath"]).reset_index(drop=True)

manifest.to_csv(OUT_DIR / "manifest.csv", index=False)
print("Manifest rows after join:", len(manifest))

# -----------------------------
# 3) STANDARD PATIENT-LEVEL SPLIT
# -----------------------------
train_p, val_p, test_p = stratified_patient_split(patients, seed=RNG_SEED)

def assign_split(pid):
    if pid in train_p: return "train"
    if pid in val_p: return "val"
    if pid in test_p: return "test"
    return "unassigned"

manifest["split_standard"] = manifest["Patient"].apply(assign_split)
manifest.to_csv(OUT_DIR / "manifest_with_standard_split.csv", index=False)

# Split files (image-level)
manifest[manifest["split_standard"]=="train"].to_csv(OUT_DIR / "split_standard_train.csv", index=False)
manifest[manifest["split_standard"]=="val"].to_csv(OUT_DIR / "split_standard_val.csv", index=False)
manifest[manifest["split_standard"]=="test"].to_csv(OUT_DIR / "split_standard_test.csv", index=False)

# -----------------------------
# 4) MODALITY SHIFT SPLITS (only if modality exists)
# -----------------------------
has_modality = manifest["modality"].notna().mean() > 0.5  # threshold; adjust
print("Modality coverage:", manifest["modality"].notna().mean())

if has_modality:
    # WLE -> NBI
    wle_train = manifest[(manifest["modality"]=="WLE")].copy()
    nbi_test  = manifest[(manifest["modality"]=="NBI")].copy()

    wle_train.to_csv(OUT_DIR / "split_WLE_to_NBI_train.csv", index=False)
    nbi_test.to_csv(OUT_DIR / "split_WLE_to_NBI_test.csv", index=False)

    # NBI -> WLE
    nbi_train = manifest[(manifest["modality"]=="NBI")].copy()
    wle_test  = manifest[(manifest["modality"]=="WLE")].copy()

    nbi_train.to_csv(OUT_DIR / "split_NBI_to_WLE_train.csv", index=False)
    wle_test.to_csv(OUT_DIR / "split_NBI_to_WLE_test.csv", index=False)

print("Artifacts written to:", OUT_DIR)


Patients: 210 | Malignant: 59 | Benign: 151
Images found (mappable): 11157
Manifest rows after join: 11157
Modality coverage: 0.0
Artifacts written to: /content/drive/MyDrive/ENT/Data/_challenge2_artifacts


In [ ]:
import os
from pathlib import Path
import pandas as pd
from PIL import Image
from collections import Counter
import random

RNG_SEED = 42
random.seed(RNG_SEED)

ART_DIR = Path(r"/content/drive/MyDrive/ENT/Data/_challenge2_artifacts")
MANIFEST_PATH = ART_DIR / "manifest.csv"
OUT_DIR = ART_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------ load manifest ------------
df = pd.read_csv(MANIFEST_PATH)
assert {"filepath","Patient","label_binary","Leukoplakia"}.issubset(df.columns), "manifest.csv missing required columns"

# ------------ read resolution (fast & safe) ------------
def get_hw(fp: str):
    try:
        with Image.open(fp) as im:
            w, h = im.size
        return h, w
    except Exception:
        return None, None

# Avoid re-reading if you already ran this before
if "height" not in df.columns or "width" not in df.columns:
    heights, widths = [], []
    bad = 0
    for fp in df["filepath"].tolist():
        h, w = get_hw(fp)
        if h is None:
            bad += 1
        heights.append(h)
        widths.append(w)
    df["height"] = heights
    df["width"] = widths
    print(f"Resolution read complete. Unreadable images: {bad}")
else:
    print("height/width already present. Skipping image reads.")

df["res"] = df["width"].astype(str) + "x" + df["height"].astype(str)

# Drop rows with missing resolution (should be ~0)
df = df[df["height"].notna() & df["width"].notna()].copy()

# ------------ define resolution groups ------------
# Group by exact resolution; pick the majority resolution as "major"
res_counts = Counter(df["res"].tolist())
res_counts_df = pd.DataFrame({"res": list(res_counts.keys()), "count": list(res_counts.values())}).sort_values("count", ascending=False)

major_res = res_counts_df.iloc[0]["res"]
print("Major resolution:", major_res, "| Count:", int(res_counts_df.iloc[0]["count"]))
print("Top 10 resolutions:\n", res_counts_df.head(10).to_string(index=False))

df["res_group"] = df["res"].apply(lambda r: "major" if r == major_res else "minor")

# Save manifest with resolution metadata
df.to_csv(OUT_DIR / "manifest_with_resolution.csv", index=False)
res_counts_df.to_csv(OUT_DIR / "resolution_counts.csv", index=False)

# ------------ build patient-level splits ------------
# Standard patient-level split (stratified)
patients = df[["Patient","label_binary"]].drop_duplicates().copy()

benign = patients[patients["label_binary"]==0]["Patient"].tolist()
malig  = patients[patients["label_binary"]==1]["Patient"].tolist()

random.shuffle(benign)
random.shuffle(malig)

def split_list(lst, train=0.70, val=0.10):
    n = len(lst)
    n_train = int(round(n*train))
    n_val   = int(round(n*val))
    tr = lst[:n_train]
    va = lst[n_train:n_train+n_val]
    te = lst[n_train+n_val:]
    return tr, va, te

b_tr,b_va,b_te = split_list(benign)
m_tr,m_va,m_te = split_list(malig)

train_p = set(b_tr + m_tr)
val_p   = set(b_va + m_va)
test_p  = set(b_te + m_te)

def assign_standard(pid):
    if pid in train_p: return "train"
    if pid in val_p: return "val"
    if pid in test_p: return "test"
    return "unassigned"

df["split_standard"] = df["Patient"].apply(assign_standard)

# Save standard split files
df[df["split_standard"]=="train"].to_csv(OUT_DIR / "split_standard_train.csv", index=False)
df[df["split_standard"]=="val"].to_csv(OUT_DIR / "split_standard_val.csv", index=False)
df[df["split_standard"]=="test"].to_csv(OUT_DIR / "split_standard_test.csv", index=False)

# ------------ resolution-shift benchmark ------------
# Train/val only on major resolution; test only on minor resolution
df_major = df[df["res_group"]=="major"].copy()
df_minor = df[df["res_group"]=="minor"].copy()

# Patient-level: ensure no patient leaks between major-train and minor-test
major_patients = set(df_major["Patient"].unique().tolist())
minor_patients = set(df_minor["Patient"].unique().tolist())

# Patients that have images in both groups create leakage risk; exclude them from the shift benchmark
overlap_patients = major_patients.intersection(minor_patients)
print("Patients with both major+minor resolutions (excluded from shift benchmark):", len(overlap_patients))

df_major_shift = df_major[~df_major["Patient"].isin(overlap_patients)].copy()
df_minor_shift = df_minor[~df_minor["Patient"].isin(overlap_patients)].copy()

# Split major_shift into train/val (patient-level stratified)
p_shift = df_major_shift[["Patient","label_binary"]].drop_duplicates()
ben = p_shift[p_shift["label_binary"]==0]["Patient"].tolist()
mal = p_shift[p_shift["label_binary"]==1]["Patient"].tolist()

random.shuffle(ben); random.shuffle(mal)

# Use 85/15 for train/val since test is separate
b_tr2,b_va2,_ = split_list(ben, train=0.85, val=0.15)
m_tr2,m_va2,_ = split_list(mal, train=0.85, val=0.15)

train_shift_p = set(b_tr2 + m_tr2)
val_shift_p   = set(b_va2 + m_va2)

df_major_shift["split_res_shift"] = df_major_shift["Patient"].apply(lambda pid: "train" if pid in train_shift_p else ("val" if pid in val_shift_p else "drop"))
df_minor_shift["split_res_shift"] = "test"

# Write resolution shift splits
df_major_shift[df_major_shift["split_res_shift"]=="train"].to_csv(OUT_DIR / "split_res_shift_train_major.csv", index=False)
df_major_shift[df_major_shift["split_res_shift"]=="val"].to_csv(OUT_DIR / "split_res_shift_val_major.csv", index=False)
df_minor_shift.to_csv(OUT_DIR / "split_res_shift_test_minor.csv", index=False)

# ------------ leukoplakia hard-subgroup test ------------
# Build a test subset from the STANDARD test split
df_test = df[df["split_standard"]=="test"].copy()
df_test_leuko = df_test[df_test["Leukoplakia"].astype(str).str.lower().isin(["yes","y","true","1"])].copy()
df_test_nonleuko = df_test[~df_test.index.isin(df_test_leuko.index)].copy()

df_test_leuko.to_csv(OUT_DIR / "test_subgroup_leukoplakia.csv", index=False)
df_test_nonleuko.to_csv(OUT_DIR / "test_subgroup_nonleukoplakia.csv", index=False)

print("STANDARD test images:", len(df_test))
print("Leukoplakia subgroup images:", len(df_test_leuko))
print("Non-leukoplakia subgroup images:", len(df_test_nonleuko))

print("Done. Artifacts in:", OUT_DIR)


Resolution read complete. Unreadable images: 0
Major resolution: 1280x1008 | Count: 6714
Top 10 resolutions:
       res  count
1280x1008   6714
  868x540   1476
1736x1080   1266
  720x544   1153
1842x1080    548
Patients with both major+minor resolutions (excluded from shift benchmark): 0
STANDARD test images: 2438
Leukoplakia subgroup images: 329
Non-leukoplakia subgroup images: 2109
Done. Artifacts in: /content/drive/MyDrive/ENT/Data/_challenge2_artifacts
